In [17]:
# initial prompt draft
# eval dataset
# feed through claude
# feed them through a grader (maybe out of 10)
# avg scores
# change prompt in some way and repeat above steps

from dotenv import load_dotenv
import os

load_dotenv()

# create an API client
# from anthropic import Anthropic
from openai import OpenAI

# client = Anthropic()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

model = "anthropic/claude-opus-4.6"

def add_user_message(messages, text):
    user_msg = {"role": "user", "content": text}
    messages.append(user_msg)

def add_assistant_message(messages, text):
    assistant_msg = {"role": "assistant", "content": text}
    messages.append(assistant_msg)

def chat(messages):
    message = client.chat.completions.create(
        model=model,
        max_tokens=90,
        messages=messages,
        # stop=stop
    )
    return message.choices[0].message.content

In [21]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria for Evaluation of solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>


Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages)
    
    # Extract JSON from response
    text = text.strip()
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
    
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"Failed to parse JSON: {e}")
        print(f"Raw text: {text[:200]}")
        # Return default values on parse failure
        return {
            "strengths": ["Unable to evaluate"],
            "weaknesses": ["Evaluation failed"],
            "reasoning": f"Parse error: {str(e)}",
            "score": 5
        }

In [22]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [23]:
from statistics import mean
def run_prompt(test_case):
    # merges the prompt and test case and returns the result
    prompt_v1 = f"""
    please solve the following task:
    {test_case["task"]}
    * respond only with python, json or a plain regex
    * do not include any explanations, comments, or additional text
    """
    messages=[]
    add_user_message(messages,prompt_v1)
    add_assistant_message(messages, "```code") # code block to indicate the start of the response
    text = chat(messages)
    text = text.strip()
    if "```code" in text:
        text = text.split("```code")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
        
    return text
    # pass


def run_test_case(test_case):
    # calls the run_prompt,then grades the result
    output = run_prompt(test_case)
    # todo grade
    model_eval = grade_by_model(test_case, output,)
    model_score = model_eval["score"]
    reasoning = model_eval["reasoning"]

    syntax_score = grade_syntax(output, test_case)
    score = (model_score + syntax_score) / 2  # Average the model score and syntax score
    # score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }
    # pass

def run_eval(dataset):
    # loads the dataset and calls run_test_case for each case
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score across {len(results)} test cases: {average_score:.2f}")

    return results
    # pass

In [24]:
import json
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

APIStatusError: Error code: 402 - {'error': {'message': 'This request requires more credits, or fewer max_tokens. You requested up to 90 tokens, but can only afford 73. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account', 'code': 402, 'metadata': {'limit_source': 'openrouter_credits', 'remedy_hint': 'Add credits at https://openrouter.ai/settings/credits, or lower max_tokens / prompt size to fit your remaining balance.', 'provider_name': None, 'previous_errors': [{'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 90 tokens, but can only afford 73. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request requires more credits, or fewer max_tokens. You requested up to 90 tokens, but can only afford 73. To increase, visit https://openrouter.ai/settings/credits and upgrade to a paid account'}, {'code': 402, 'message': 'This request